# Oncology Patient Flow & Care Quality — EDA
**Project:** EMR Clinical Analytics | **Data:** Synthea Synthetic EMR  
**Purpose:** Exploratory analysis of patient encounters, diagnoses, and quality metrics to surface insights for clinical leadership.

---
### Notebook structure
1. Setup & data loading
2. Data quality checks
3. Joining encounters + conditions + patients
4. Summary statistics
5. Key clinical metrics (volume, LOS, readmissions, HbA1c)
6. Visualizations
7. Export for Tableau

## 1. Setup & data loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 4)})

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR = '../data/raw/'        # folder where Synthea CSVs live
OUT_DIR  = '../data/processed/'  # cleaned outputs for Tableau
os.makedirs(OUT_DIR, exist_ok=True)

print('Libraries loaded ✓')

In [ ]:
# ── Load core Synthea tables ──────────────────────────────────────────────────
patients    = pd.read_csv(DATA_DIR + 'patients.csv',    low_memory=False)
encounters  = pd.read_csv(DATA_DIR + 'encounters.csv',  low_memory=False)
conditions  = pd.read_csv(DATA_DIR + 'conditions.csv',  low_memory=False)
observations= pd.read_csv(DATA_DIR + 'observations.csv',low_memory=False)
medications = pd.read_csv(DATA_DIR + 'medications.csv', low_memory=False)

for name, df in [('patients', patients), ('encounters', encounters),
                 ('conditions', conditions), ('observations', observations),
                 ('medications', medications)]:
    print(f'{name:15s}: {df.shape[0]:>7,} rows  x  {df.shape[1]:>3} cols')

## 2. Data quality checks
Before analysis, we inspect missingness, duplicate IDs, and date validity — the same checks expected when working with real Epic EMR exports.

In [ ]:
def quality_report(df, name):
    """Print null rates and basic dtype summary for a dataframe."""
    print(f'\n{'='*55}')
    print(f'  {name.upper()} — quality report')
    print(f'{'='*55}')
    nulls = df.isnull().mean().mul(100).round(1)
    nulls = nulls[nulls > 0].sort_values(ascending=False)
    if nulls.empty:
        print('  No missing values ✓')
    else:
        print(nulls.to_string(header=False))
    dupes = df.duplicated().sum()
    print(f'  Duplicate rows: {dupes}')

for name, df in [('patients', patients), ('encounters', encounters),
                 ('conditions', conditions)]:
    quality_report(df, name)

In [ ]:
# ── Parse dates ───────────────────────────────────────────────────────────────
encounters['START'] = pd.to_datetime(encounters['START'])
encounters['STOP']  = pd.to_datetime(encounters['STOP'])
patients['BIRTHDATE'] = pd.to_datetime(patients['BIRTHDATE'])
patients['DEATHDATE'] = pd.to_datetime(patients['DEATHDATE'])
conditions['START'] = pd.to_datetime(conditions['START'])
conditions['STOP']  = pd.to_datetime(conditions['STOP'])

# Check for any encounters with STOP before START (data integrity flag)
bad_dates = encounters[encounters['STOP'] < encounters['START']]
print(f'Encounters with STOP < START: {len(bad_dates)}  (expected: 0)')

# Check date range of data
print(f'Encounter date range: {encounters["START"].min().date()} → {encounters["START"].max().date()}')

## 3. Data cleaning & feature engineering

In [ ]:
# ── Patient demographics ───────────────────────────────────────────────────────
ref_date = encounters['START'].max()  # use last encounter as reference date

patients['age'] = ((ref_date - patients['BIRTHDATE']).dt.days / 365.25).astype(int)
patients['deceased'] = patients['DEATHDATE'].notna()

# Clean column names to lowercase
patients.columns = patients.columns.str.lower()
encounters.columns = encounters.columns.str.lower()
conditions.columns = conditions.columns.str.lower()
observations.columns = observations.columns.str.lower()
medications.columns = medications.columns.str.lower()

print('Demographics added ✓')
print(patients[['id','age','gender','race','ethnicity','deceased']].head(3).to_string(index=False))

In [ ]:
# ── Encounter feature engineering ─────────────────────────────────────────────
# Length of stay in hours and days
encounters['los_hours'] = (encounters['stop'] - encounters['start']).dt.total_seconds() / 3600
encounters['los_days']  = encounters['los_hours'] / 24

# Encounter year and month (for trending)
encounters['enc_year']  = encounters['start'].dt.year
encounters['enc_month'] = encounters['start'].dt.to_period('M').astype(str)

# Flag inpatient encounters (LOS >= 1 day)
encounters['is_inpatient'] = encounters['los_days'] >= 1

print('Encounter features added ✓')
print(encounters[['id','start','stop','encounterclass','los_hours','los_days','is_inpatient']].head(3).to_string(index=False))

In [ ]:
# ── Oncology filter — ICD-10 C codes (malignant neoplasms) ────────────────────
# Synthea uses SNOMED; we filter by description keywords as a proxy.
# In Epic/real EMR, you would filter by ICD-10 C18-C80 range.

CANCER_KEYWORDS = [
    'malignant', 'carcinoma', 'lymphoma', 'leukemia',
    'neoplasm', 'cancer', 'tumor', 'sarcoma', 'myeloma'
]
pattern = '|'.join(CANCER_KEYWORDS)

cancer_conditions = conditions[
    conditions['description'].str.lower().str.contains(pattern, na=False)
].copy()

cancer_patient_ids = cancer_conditions['patient'].unique()
print(f'Oncology patients identified: {len(cancer_patient_ids):,}')
print(f'\nTop 10 oncology diagnoses:')
print(cancer_conditions['description'].value_counts().head(10).to_string())

## 4. Joining encounters + conditions + patients
This mirrors the table joins a clinical analyst performs against Epic's Clarity (relational) database.

In [ ]:
# ── Build the master analytical table ─────────────────────────────────────────
# Step 1: Link conditions to encounters (on patient + encounter)
enc_conditions = encounters.merge(
    conditions[['encounter','description','code']].rename(columns={
        'description': 'diagnosis',
        'code': 'diagnosis_code'
    }),
    left_on='id', right_on='encounter',
    how='left'
)

# Step 2: Attach patient demographics
master = enc_conditions.merge(
    patients[['id','age','gender','race','ethnicity','deceased','city','state']].rename(
        columns={'id': 'patient_id'}
    ),
    left_on='patient', right_on='patient_id',
    how='left'
)

# Step 3: Flag oncology encounters
master['is_oncology'] = master['patient'].isin(cancer_patient_ids)

# Step 4: Flag oncology-related diagnosis on the row
master['is_cancer_dx'] = master['diagnosis'].str.lower().str.contains(pattern, na=False)

print(f'Master table: {master.shape[0]:,} rows x {master.shape[1]} columns')
print(f'Oncology encounters: {master[master["is_oncology"]].shape[0]:,}')
master.head(3)

In [ ]:
# ── Oncology subset ────────────────────────────────────────────────────────────
onc = master[master['is_oncology']].copy()
print(f'Oncology master table: {onc.shape[0]:,} rows, {onc["patient"].nunique():,} unique patients')

## 5. Summary statistics

In [ ]:
# ── Patient demographics summary ───────────────────────────────────────────────
onc_pts = patients[patients['id'].isin(cancer_patient_ids)]

print('=== Oncology Patient Demographics ===')
print(f'  Total patients          : {len(onc_pts):,}')
print(f'  Median age              : {onc_pts["age"].median():.0f} years')
print(f'  Age range               : {onc_pts["age"].min()}–{onc_pts["age"].max()}')
print(f'  % Female                : {(onc_pts["gender"]=="F").mean()*100:.1f}%')
print(f'  % Deceased              : {onc_pts["deceased"].mean()*100:.1f}%')
print()
print('Gender breakdown:')
print(onc_pts['gender'].value_counts(normalize=True).mul(100).round(1).to_string())
print()
print('Race breakdown:')
print(onc_pts['race'].value_counts(normalize=True).mul(100).round(1).head(6).to_string())

In [ ]:
# ── Encounter class breakdown ──────────────────────────────────────────────────
print('=== Encounter Types (Oncology Patients) ===')
enc_class = onc.drop_duplicates('id')['encounterclass'].value_counts()
print(enc_class.to_string())

print('\n=== Annual Encounter Volume ===')
annual = onc.drop_duplicates('id').groupby('enc_year').size().reset_index(name='encounter_count')
print(annual.to_string(index=False))

In [ ]:
# ── Length of stay analysis ────────────────────────────────────────────────────
inpatient = onc[(onc['is_inpatient']) & (onc['los_days'] < 365)].drop_duplicates('id')

print('=== Inpatient Length of Stay (days) ===')
print(inpatient['los_days'].describe().round(2).to_string())
print(f'\nMedian LOS : {inpatient["los_days"].median():.1f} days')
print(f'Mean LOS   : {inpatient["los_days"].mean():.1f} days')
print(f'>7 day LOS : {(inpatient["los_days"]>7).mean()*100:.1f}% of inpatient encounters')

In [ ]:
# ── 30-day readmission flag ────────────────────────────────────────────────────
# Sort each patient's encounters by start date, then check if next encounter
# starts within 30 days of prior encounter stop.

readm = (
    onc[onc['is_inpatient']]
    .drop_duplicates('id')
    .sort_values(['patient', 'start'])
    [['patient','start','stop','id']]
    .copy()
)

readm['prev_stop']      = readm.groupby('patient')['stop'].shift(1)
readm['days_since_d/c'] = (readm['start'] - readm['prev_stop']).dt.days
readm['readmit_30d']    = (readm['days_since_d/c'] > 0) & (readm['days_since_d/c'] <= 30)

readmit_rate = readm['readmit_30d'].mean() * 100
print(f'30-day readmission rate (inpatient): {readmit_rate:.1f}%')
print(f'Total readmissions identified: {readm["readmit_30d"].sum()}')

In [ ]:
# ── HbA1c quality metric (diabetes follow-up) ─────────────────────────────────
# In oncology, diabetes management is a key comorbidity quality metric.
# LOINC 4548-4 = HbA1c

hba1c = observations[
    (observations['code'] == '4548-4') |
    (observations['description'].str.contains('Hemoglobin A1c', case=False, na=False))
].copy()

hba1c['value_numeric'] = pd.to_numeric(hba1c['value'], errors='coerce')
hba1c_onc = hba1c[hba1c['patient'].isin(cancer_patient_ids)]

print(f'HbA1c observations in oncology patients: {len(hba1c_onc):,}')
print(f'Patients with at least one HbA1c: {hba1c_onc["patient"].nunique():,}')

if len(hba1c_onc) > 0:
    print(f'\nHbA1c value distribution:')
    print(hba1c_onc['value_numeric'].describe().round(2).to_string())
    pct_controlled = (hba1c_onc['value_numeric'] < 7).mean() * 100
    print(f'\n% with HbA1c < 7% (controlled): {pct_controlled:.1f}%')
    print(f'% with HbA1c ≥ 9% (poor control): {(hba1c_onc["value_numeric"]>=9).mean()*100:.1f}%')

## 6. Visualizations
Production-ready charts formatted for a clinical leadership audience.

In [ ]:
# ── 6.1 Annual encounter volume trend ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 4))

vol = onc.drop_duplicates('id').groupby('enc_year').size().reset_index(name='count')
ax.bar(vol['enc_year'], vol['count'], color='#2E86AB', edgecolor='white', linewidth=0.5)
ax.set_title('Annual Oncology Encounter Volume', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Year')
ax.set_ylabel('Encounter count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
sns.despine()
plt.tight_layout()
plt.savefig('../figures/01_encounter_volume.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 6.2 Encounter class breakdown ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))

enc_c = onc.drop_duplicates('id')['encounterclass'].value_counts()
colors = sns.color_palette('muted', len(enc_c))
bars = ax.barh(enc_c.index, enc_c.values, color=colors, edgecolor='white')

for bar, val in zip(bars, enc_c.values):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=10)

ax.set_title('Oncology Encounters by Type', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Number of encounters')
sns.despine()
plt.tight_layout()
plt.savefig('../figures/02_encounter_types.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 6.3 Length of stay distribution ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

los_data = inpatient['los_days'].clip(upper=30)  # cap at 30 for readability

axes[0].hist(los_data, bins=30, color='#2E86AB', edgecolor='white', linewidth=0.5)
axes[0].axvline(los_data.median(), color='#E84855', linestyle='--', linewidth=1.5,
                label=f'Median: {los_data.median():.1f}d')
axes[0].set_title('Inpatient LOS Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Length of stay (days)')
axes[0].set_ylabel('Encounters')
axes[0].legend()

# LOS by encounter class
los_by_class = onc[onc['los_days'] < 30].drop_duplicates('id')
class_order = los_by_class.groupby('encounterclass')['los_days'].median().sort_values(ascending=False).index
sns.boxplot(data=los_by_class, x='encounterclass', y='los_days',
            order=class_order, palette='muted', ax=axes[1], linewidth=0.8)
axes[1].set_title('LOS by Encounter Type', fontsize=12, fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Length of stay (days)')
axes[1].tick_params(axis='x', rotation=30)

sns.despine()
plt.tight_layout()
plt.savefig('../figures/03_length_of_stay.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 6.4 Top oncology diagnoses ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))

top_dx = cancer_conditions['description'].value_counts().head(12)
colors_dx = ['#2E86AB' if i < 3 else '#8FB8C9' for i in range(len(top_dx))]

bars = ax.barh(top_dx.index[::-1], top_dx.values[::-1], color=colors_dx[::-1], edgecolor='white')
for bar, val in zip(bars, top_dx.values[::-1]):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)

ax.set_title('Top 12 Oncology Diagnoses', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Number of patients')
sns.despine()
plt.tight_layout()
plt.savefig('../figures/04_top_diagnoses.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 6.5 Patient age distribution by gender ────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))

for gender, color, label in [('F','#E84855','Female'), ('M','#2E86AB','Male')]:
    subset = onc_pts[onc_pts['gender'] == gender]['age']
    ax.hist(subset, bins=20, alpha=0.6, color=color, label=label, edgecolor='white')

ax.set_title('Age Distribution — Oncology Patients', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Age (years)')
ax.set_ylabel('Number of patients')
ax.legend()
sns.despine()
plt.tight_layout()
plt.savefig('../figures/05_age_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 6.6 HbA1c control status ──────────────────────────────────────────────────
if len(hba1c_onc) > 0 and hba1c_onc['value_numeric'].notna().sum() > 0:
    fig, ax = plt.subplots(figsize=(7, 4))

    # Get most recent HbA1c per patient
    latest_hba1c = (
        hba1c_onc.sort_values('date')
        .groupby('patient')
        .last()
        .reset_index()
    )
    latest_hba1c['control_status'] = pd.cut(
        latest_hba1c['value_numeric'],
        bins=[0, 7, 9, 100],
        labels=['Controlled (<7%)', 'At risk (7–9%)', 'Poor control (≥9%)']
    )
    ctrl = latest_hba1c['control_status'].value_counts()

    colors_hba1c = ['#3BB273', '#F4A261', '#E84855']
    wedges, texts, autotexts = ax.pie(
        ctrl.values, labels=ctrl.index,
        colors=colors_hba1c, autopct='%1.1f%%',
        startangle=90, pctdistance=0.75
    )
    for t in autotexts:
        t.set_fontsize(10)

    ax.set_title('HbA1c Control Status\n(Most recent value, oncology patients)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../figures/06_hba1c_control.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No HbA1c data found — skipping chart. Check LOINC code or observations file.')

## 7. Export for Tableau
Three clean CSVs are written to `data/processed/` for direct connection in Tableau.

In [ ]:
# ── Export 1: Encounter-level flat table (for Tableau dashboards) ─────────────
tableau_encounters = onc.drop_duplicates('id')[[
    'id', 'patient', 'start', 'stop', 'enc_year', 'enc_month',
    'encounterclass', 'los_hours', 'los_days', 'is_inpatient',
    'age', 'gender', 'race', 'ethnicity', 'city', 'state',
    'is_oncology', 'is_cancer_dx'
]].copy()

tableau_encounters.to_csv(OUT_DIR + 'encounters_clean.csv', index=False)
print(f'encounters_clean.csv saved: {len(tableau_encounters):,} rows')

# ── Export 2: Diagnosis frequency table ───────────────────────────────────────
dx_freq = (
    cancer_conditions
    .groupby(['description','code'])
    .agg(patient_count=('patient','nunique'),
         encounter_count=('encounter','nunique'))
    .reset_index()
    .sort_values('patient_count', ascending=False)
)
dx_freq.to_csv(OUT_DIR + 'diagnosis_frequency.csv', index=False)
print(f'diagnosis_frequency.csv saved: {len(dx_freq):,} rows')

# ── Export 3: Readmission summary ─────────────────────────────────────────────
readm_export = readm[['patient','id','start','stop','days_since_d/c','readmit_30d']]
readm_export.to_csv(OUT_DIR + 'readmissions.csv', index=False)
print(f'readmissions.csv saved: {len(readm_export):,} rows')

# ── Export 4: HbA1c summary (if available) ────────────────────────────────────
if len(hba1c_onc) > 0:
    hba1c_onc[['patient','date','code','description','value','value_numeric']].to_csv(
        OUT_DIR + 'hba1c_observations.csv', index=False
    )
    print(f'hba1c_observations.csv saved: {len(hba1c_onc):,} rows')

print('\n✓ All exports complete. Connect Tableau to data/processed/')

## Summary of key findings

| Metric | Value |
|---|---|
| Total oncology patients | *(run cells above)* |
| Median patient age | *(run cells above)* |
| Median inpatient LOS | *(run cells above)* |
| 30-day readmission rate | *(run cells above)* |
| % HbA1c controlled | *(run cells above)* |

> **Next step:** Open Tableau, connect to `data/processed/encounters_clean.csv` and build the performance dashboard (see `notebooks/02_tableau_prep.ipynb`).